# SAHYOG - SAR Detector Training (GPU) + QNN / TFLite Export
**SIH 2026 · PS 26177 · Robotics & Drones**

Runs the full-size training the 2-core sandbox cannot: YOLO11n survivor detector
(RGB + LWIR, P2 stride-4 head) fine-tuned on real aerial SAR datasets, exported to
ONNX (flight), QNN (Qualcomm RB3 Gen 2 NPU) and TFLite-INT8 (phone Hexagon NPU).

Without a GPU, `scripts/train_detector.py --synthesize` renders a labelled dataset from the
radiometric simulator and a CPU trains a small YOLOv8n on it (that produced models/yolov8n-*-sim.onnx).
This notebook is the GPU-scale path on real data.


## 0 · Why a GPU, and what changes on the aircraft
- Training needs a GPU; inference runs on 12 TOPS (RB3 Gen 2) or the phone NPU.
- A quantisation GATE ships an INT8 model only if it retains >=97% recall vs FP32 (scripts/export_model.py --validate).
- The aircraft never downloads weights; fetch_models.py is a deliberate human step.


## 1 · Clone the repo + install the training stack


In [ ]:
import os
os.chdir('/content')
!git clone https://github.com/ShashankRoy-iit/SIH.git
os.chdir('/content/SIH')
!pip install -q -e '.[sim,dev]'
!pip install -q ultralytics onnx onnxruntime
import torch, ultralytics
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| ultralytics', ultralytics.__version__)


## 2 · Get a real dataset (pick one)
Hard case = small, low-contrast aerial thermal persons. Best public matches:
- HIT-UAV (2,898 aerial LWIR frames) - GitHub
- AIResQ (high-res airborne thermal SAR, best published aerial-thermal mAP) - Nature Sci Data
- SARD (search-and-rescue postures, RGB) - IEEE DataPort
Or synthesise a starting set from the simulator (no download):


In [ ]:
# 4k thermal frames rendered from the radiometric simulator
!python scripts/train_detector.py --synthesize 4000 --scenario mixed --modality lwir --alt-min 30 --alt-max 90 --out datasets/sim-thermal
# once HIT-UAV is converted to YOLO format under datasets/hit-uav/:
!python scripts/train_detector.py --dataset-yaml hit-uav --out datasets/


## 3 · Train (GPU)
--p2 adds the stride-4 head (8 px person vs missing them). Thermal uses a 1-channel stem; RGB stays 3-channel.


In [ ]:
# Thermal (LWIR), P2 head - the deployment detector
!python scripts/train_detector.py --train --data datasets/sim-thermal/data.yaml --model yolo11n.yaml --p2 --channels 1 --imgsz 640 --epochs 120 --batch 16 --name thermal --device 0

# RGB (phone camera)
!python scripts/train_detector.py --train --data datasets/sim-rgb/data.yaml --model yolo11n.yaml --imgsz 640 --epochs 120 --batch 16 --name rgb --device 0


## 4 · Export to the formats the aircraft actually runs


In [ ]:
# FP32 ONNX (opset 12 keeps the Qualcomm QNN converter happy)
!python scripts/export_model.py --weights runs/sar/thermal/weights/best.pt --out models/yolo11n-thermal-sar.onnx --imgsz 640 512 --channels 1
!python scripts/export_model.py --weights runs/sar/rgb/weights/best.pt --out models/yolo11n-rgb-sim.onnx --imgsz 640 640 --channels 3

# Phone NPU: TFLite full-integer (INT8) recipe - RGB model runs on the phone Hexagon DSP
!python scripts/export_phone_model.py --recipe


## 5 · Quantise for the Qualcomm NPU (RB3 Gen 2 / QCS6490)
Needs a Qualcomm AI Hub API token (free). Emits a .bin QNN model library.


In [ ]:
!pip install -q 'qai-hub-models[yolov11_det]' qai-hub
!qai-hub configure --api_token YOUR_TOKEN_HERE
!python -m qai_hub_models.models.yolov11_det.export --quantize w8a8 --target-runtime qnn --chipset qualcomm-qcs6490-proxy --output-dir models/
# the gate: INT8 must retain >=97% recall vs FP32 on deployment-statistics frames
!python scripts/export_model.py --validate --reference models/yolo11n-thermal-sar.onnx --candidate models/yolo11n-thermal-sar-qcs6490.bin --frames 60 --json artifacts/qnn_validation.json


## 6 · Register + verify + fly
Checksums prove the model on disk is the model that produced a measured result.


In [ ]:
!python scripts/fetch_models.py --verify-all
!python scripts/eval_detector.py --mode both --json artifacts/detector_eval_neural.json
!SAR_DETECTOR=auto python scripts/run_mission.py --scenario flood --duration 460 --live


## 7 · Expected results (published / measured baselines)
- Aerial-thermal person detection: published mAP50 ~0.55 (AIResQ-trained YOLO) is the honest ceiling; small objects are the hard subset.
- The repo's heuristic ensemble already reaches recall 1.00 of resolvable targets at 35/50/70 m; the neural model's job is precision (rejecting rooftops/vehicles).
- Ship only what the quantisation gate passes.


---
Back on the aircraft: drop .onnx/.bin into models/, set detector: auto in configs/onboard.yaml,
and put the phone RGB TFLite into the Termux bridge (mobile/phone_onboard.py). The stack picks them up automatically.
